One of the first things you learn when applying machine learning models is the notion of cross-validation. Training a model, which basically is learning the parameters of a prediction function, and evaluating the performance of a model on the same dataset is a methodological mistake.

The machine learning model could learn the labels of the data and reproduce them. However, this is not really what we want. If we then deploy the machine learning model on unseen data, we run the risk of predicting not anything useful. Our model has only learned the training data.

::: {.column-margin}
::: {.callout-note}
To demonstrate this technique, we use the [New York City Taxi Trip Duration](https://www.kaggle.com/c/nyc-taxi-trip-duration/overview) competition hosted on [Kaggle](https://www.kaggle.com/). In this competition, we try to predict the trip duration.
:::
:::

In this post, I present a robust cross-validation method for tabular data. This method, among other things, can be used in Data Science competitions.

This post is not intended to achieve a high score in the competition in question. I  demonstrate a way to implement a cross-validation method that gives the same results on validation data as on unseen test data.

## $k$-fold cross-validation

We load the train and test data into the cell above. I drop the `dropoff_datetime` because this feature isn't present in the testing data.

We use $k$-fold cross-validation to implement our cross-validation method. In $k$-fold cross-validation we estimate what our model has learned based on new data that the model has not yet seen. The $k$ in $k$-fold cross-validation shows how many groups we divide our data into. If $k$ = 5 then we divide our data into 5 groups. We then train our algorithm on 4 groups of the data, and we validate the model on the 1 group of data that the model has not yet seen. We repeat this process until we have trained our model on each group of data once. Also, each group of data has been used to validate our model.

To implement this method we use `sklearn`'s $k$-fold cross-validation capabilities. In our case, we are dealing with a regression problem. This is why we use a simple k-fold method. However, if you are dealing with a classification problem you can use stratified $k$-fold cross-validation. This ensures that the frequency of labels between the folds is the same. And increases ensures better consistency between validation and testing scores.


In [1]:
import pandas as pd
from sklearn.model_selection import KFold

# number of folds used
FOLDS = 5

# instantiate k-fold cross-validation
kfold = KFold(n_splits=FOLDS, shuffle=False)

In this case choose $k$ = 5. We have a relatively large data set, so it seems to me that 5 folds will generate a robust validation score. We are now going to divide our data into 5 groups/folds. We create a new column in which we record the fold the observation belongs to.

However, before we do this, we randomize the order of the data. This way, we prevent the order in which the data is placed from becoming important, which can incorporate bias in our validation score.


In [3]:
# load data into dataframe
df = pd.read_csv("data/taxi.csv")

In [10]:
#| column: page
# randomize order of dataframe
df = df.sample(frac=1, random_state=1, random_state=42).reset_index(drop=True)

# create fold columns and store the fold
for fold, (train_idx, valid_idx) in enumerate(kfold.split(
    X=train, y=df['trip_duration'])
):
    df.loc[valid_idx, 'kfold'] = fold

# display rows
df.head(3)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,kfold
0,id0880738,2,2016-02-27 20:13:05,2016-02-27 20:24:37,1,-73.981728,40.749500,-73.945915,40.792061,N,692,0.0
1,id2002545,2,2016-06-04 09:54:05,2016-06-04 10:10:35,1,-73.979088,40.771606,-73.946518,40.822655,N,990,0.0
2,id0289724,2,2016-05-06 17:40:05,2016-05-06 17:50:52,1,-73.989700,40.738651,-73.997772,40.754051,N,647,0.0



Now we have created a column in which we record the fold the observation belongs to. If we look at the size distribution of the number of folds, we see that the folds are approximately the same. This is what we want!

Sometimes there can be a small difference in the number of observations (in our case!) because the number of recorded samples is not easily divisible by the number of folds.

In [11]:
# size of every fold
df['kfold'].value_counts()

0.0    291729
1.0    291729
2.0    291729
3.0    291729
4.0    291728
Name: kfold, dtype: int64

Even though this post does not focus on creating a well-performing predictive model, I'll do simple feature engineering to make the performance of our model a little bit better. This makes me feel a little bit better about myself! The function below extracts some simple date features from the date feature. This also gives us some more variables to work with.

In [14]:
#| column: screen
def extract_dates(df: pd.DataFrame, date_column: str) -> pd.DataFrame:
    df = df.copy()
    
    date_attrs = ["year", "month", "dayofweek", "day", "hour", "minute"]

    # get datetime index
    df[date_column] = pd.to_datetime(df[date_column])

    # create date features
    for date_atrr in date_attrs:
        df[f"{date_column}_{date_atrr}"] = getattr(df[date_column].dt, date_atrr)

    return df.drop([date_column], axis=1)

# extract date features
train = extract_dates(df, 'pickup_datetime')

# display
train.head(3)

,id,vendor_id,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,kfold,pickup_datetime_year,pickup_datetime_month,pickup_datetime_dayofweek,pickup_datetime_day,pickup_datetime_hour,pickup_datetime_minute
0,id0880738,2,2016-02-27 20:24:37,1,-73.981728,40.749500,-73.945915,40.792061,N,692,0.0,2016,2,5,27,20,13
1,id2002545,2,2016-06-04 10:10:35,1,-73.979088,40.771606,-73.946518,40.822655,N,990,0.0,2016,6,5,4,9,54
2,id0289724,2,2016-05-06 17:50:52,1,-73.989700,40.738651,-73.997772,40.754051,N,647,0.0,2016,5,4,6,17,40



We can then take the average prediction of these five models as our final prediction.

In [5]:
# get the mean of k predictions
final_predictions = pred_df.mean(axis=1)

# display first 3 rows
final_predictions.head(3)

NameError: name 'pred_df' is not defined

We store this average prediction in our submission file and upload it to Kaggle to get our testing score. Our testing score on Kaggle is __0.41885__. The internal mean validation score was __0.4329__, which isn't too bad of a difference.


In [ ]:
# load sample submission
sample_sub = pd.read_csv('data/sample_submission.csv')

# store predictions in column
sample_sub['trip_duration'] = final_predictions

# save predictions
sample_sub.to_csv('submissions/submission.csv', index=None)



## Conclusion

In this post, I demonstrated a simple way to use cross-validation to train a model on folds and use those folds to predict on unseen test data. This results in a cross-validation and testing score that are relatively close to each other.
